<a href="https://colab.research.google.com/github/brenonsx066-ship-it/Dr.-comunnity/blob/main/Tela_inicial%2C_paciente_medico_Dr_Community.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sqlite3


conexao = sqlite3.connect('medconnect.db')
cursor = conexao.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS pacientes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL UNIQUE,
    senha TEXT NOT NULL,
    idade INTEGER NOT NULL,
    queixa TEXT NOT NULL,
    niveldedor INTEGER NOT NULL,
    status TEXT NOT NULL
)
""")
conexao.commit()



class Atendimento:
    def cadastrar_paciente(self, nome, senha, idade, queixa, niveldedor, status):
        cursor.execute("""
        INSERT INTO pacientes (nome, senha, idade, queixa, niveldedor, status)
        VALUES (?, ?, ?, ?, ?, ?)
        """, (nome, senha, idade, queixa, niveldedor, status))
        conexao.commit()

    def listar_cadastros(self):
        cursor.execute("SELECT id, nome, idade, queixa, niveldedor, status FROM pacientes")
        return cursor.fetchall()


atendimento_service = Atendimento()


def limpar_tela():
    os.system('cls' if os.name == 'nt' else 'clear')


def exibir_cabecalho(titulo):
    limpar_tela()
    print('=' * 65)
    print(f"{titulo:^65}")
    print('=' * 65)


def realizar_login():
    exibir_cabecalho("LOGIN DO PACIENTE")

    nome = input("\nDigite seu nome registrado: ").strip()
    senha = input("Digite sua senha: ").strip()


    cursor.execute("""
    SELECT id, nome, idade, queixa, niveldedor, status
    FROM pacientes
    WHERE LOWER(nome) = LOWER(?) AND senha = ?
    """, (nome, senha))

    paciente = cursor.fetchone()

    if not paciente:
        print("\nNome ou senha incorretos. Acesso negado.")
        input("\nPressione ENTER para voltar...")
        return

    p_id, p_nome, p_idade, p_queixa, p_dor, p_status = paciente


    while True:
        exibir_cabecalho(f"PAINEL DO PACIENTE - BEM-VINDO(A) {p_nome.upper()}")

        print(f"\n ESTATÍSTICAS ATUAIS:")
        print(f"    • Queixa/Sintoma : {p_queixa}")
        print(f"    • Nível de Dor   : {p_dor}/10")
        print(f"    • Status Triagem : {p_status}\n")

        print('-' * 65)
        print("  [ 1 ] Editar Estatísticas (Queixa e Nível de Dor)")
        print("  [ 2 ] Sair da conta\n")
        print('-' * 65)

        opcao = input("\nDigite o número da opção desejada: ").strip()

        if opcao == '1':
            exibir_cabecalho("EDITAR ESTATÍSTICAS DE SAÚDE")

            while True:
                nova_queixa = input("\nNova queixa/sintoma: ").strip()
                if nova_queixa:
                    break
                print("Erro. A queixa não pode ficar em branco.")

            while True:
                try:
                    novo_niveldedor = int(input("Novo nível de dor (0 a 10): "))
                    if 0 <= novo_niveldedor <= 10:
                        break
                    print("Erro. Digite um número entre 0 e 10.")
                except ValueError:
                    print("Erro. Digite apenas números inteiros.")


            novo_status = 'Urgente' if novo_niveldedor >= 7 else 'Normal'


            cursor.execute("""
            UPDATE pacientes
            SET queixa = ?, niveldedor = ?, status = ?
            WHERE id = ?
            """, (nova_queixa, novo_niveldedor, novo_status, p_id))
            conexao.commit()


            p_queixa = nova_queixa
            p_dor = novo_niveldedor
            p_status = novo_status

            print("\n" + "=" * 65)
            print("Estatísticas atualizadas com sucesso no banco de dados!")
            print("=" * 65)
            input("\nPressione ENTER para continuar...")

        elif opcao == '2':
            print("\nSaindo da conta...")
            break
        else:
            input("\nOpção inválida. Pressione ENTER para tentar novamente...")



while True:
    exibir_cabecalho("CENTRAL DE ATENDIMENTO - DR. COMMUNITY")

    print("\n  [ 1 ] Fazer um novo cadastro")
    print("  [ 2 ] Fazer login na conta")
    print("  [ 3 ] Ver cadastros realizados")
    print("  [ 4 ] Sair do programa\n")
    print('-' * 65)

    opcao = input("\nDigite o número da opção desejada: ").strip()

    match opcao:
        case '1':

            exibir_cabecalho("NOVO CADASTRO DE PACIENTE")

            while True:
                nome = input("\nQual é o seu nome completo? ").strip()
                if nome.replace(" ", "").isalpha():
                    break
                print("Erro. Digite um nome válido, sem números ou caracteres especiais.")

            while True:
                senha = input("Crie uma senha: ").strip()
                if len(senha) >= 4:
                    break
                print("Erro. A senha deve ter pelo menos 4 caracteres.")

            while True:
                try:
                    idade = int(input("Digite a sua idade? "))
                    if idade >= 0:
                        break
                    print("Erro. A idade deve ser maior ou igual a zero.")
                except ValueError:
                    print("Erro. Digite a idade apenas em números.")

            while True:
                queixa = input("Qual é a sua queixa/sintoma? ").strip()
                if queixa:
                    break
                print("Erro. A queixa não pode ficar em branco.")

            while True:
                try:
                    niveldedor = int(input("Qual é o seu nível de dor (0 a 10)? "))
                    if 0 <= niveldedor <= 10:
                        break
                    print("Erro. O nível de dor deve ser um número entre 0 e 10.")
                except ValueError:
                    print("Erro. Digite seu nível de dor apenas em números.")

            status = 'Urgente' if niveldedor >= 7 else 'Normal'

            try:

                atendimento_service.cadastrar_paciente(nome, senha, idade, queixa, niveldedor, status)
                print('\n' + '=' * 65)
                print(f"Cadastro concluído com sucesso, {nome}!")
                print(f"Grau de gravidade: {status}")
                print('=' * 65)
            except sqlite3.IntegrityError:
                print("\nErro: Já existe um cadastro com este nome. Tente fazer login.")

            input("\nPressione ENTER para voltar ao menu principal...")

        case '2':

            realizar_login()

        case '3':

            exibir_cabecalho("PACIENTES CADASTRADOS")
            registros = atendimento_service.listar_cadastros()

            if not registros:
                print("\nNenhum paciente cadastrado até o momento.")
            else:
                print(f"\n{'ID':<5} | {'NOME':<20} | {'IDADE':<6} | {'DOR':<5} | {'STATUS':<10} | QUEIXA")
                print('-' * 65)
                for paciente in registros:
                    p_id, p_nome, p_idade, p_queixa, p_dor, p_status = paciente
                    print(f"{p_id:<5} | {p_nome:<20} | {p_idade:<6} | {p_dor:<5} | {p_status:<10} | {p_queixa}")

            input("\nPressione ENTER para voltar ao menu principal...")

        case '4':

            print("\nEncerrando o sistema... Até logo!")
            break

        case _:
            input("\nOpção inválida! Pressione ENTER para tentar novamente...")


conexao.close()

             CENTRAL DE ATENDIMENTO - DR. COMMUNITY              

  [ 1 ] Fazer um novo cadastro
  [ 2 ] Fazer login na conta
  [ 3 ] Ver cadastros realizados
  [ 4 ] Sair do programa

-----------------------------------------------------------------

Digite o número da opção desejada: 1
                    NOVO CADASTRO DE PACIENTE                    

Qual é o seu nome completo? Isac de Oliveira Alexandre
Crie uma senha de acesso: 12345678
Qual é a sua idade? 23
Qual é a sua queixa/sintoma? dor de cabeça
Qual é o seu nível de dor (0 a 10)? 7

✅ Cadastro concluído com sucesso, Isac de Oliveira Alexandre!
Grau de gravidade: Urgente


In [ ]:
          try:

                atendimento_service.cadastrar_paciente(nome, senha, idade, queixa, niveldedor, status)
                print('\n' + '=' * 65)
                print(f"✅ Cadastro concluído com sucesso, {nome}!")
                print(f"Grau de gravidade: {status}")
                print('=' * 65)
            except sqlite3.IntegrityError:
                print("\n❌ Erro: Já existe um cadastro com este nome. Tente fazer login.")

            input("\nPressione ENTER para voltar ao menu principal...")

        case '2':

            realizar_login()

        case '3':

            exibir_cabecalho("PACIENTES CADASTRADOS")
            registros = atendimento_service.listar_cadastros()

            if not registros:
                print("\nNenhum paciente cadastrado até o momento.")
            else:
                print(f"\n{'ID':<5} | {'NOME':<20} | {'IDADE':<6} | {'DOR':<5} | {'STATUS':<10} | QUEIXA")
                print('-' * 65)
                for paciente in registros:
                    p_id, p_nome, p_idade, p_queixa, p_dor, p_status = paciente
                    print(f"{p_id:<5} | {p_nome:<20} | {p_idade:<6} | {p_dor:<5} | {p_status:<10} | {p_queixa}")

            input("\nPressione ENTER para voltar ao menu principal...")

        case '4':

            print("\nEncerrando o sistema... Até logo!")
            break

        case _:
            input("\n❌ Opção inválida! Pressione ENTER para tentar novamente...")


conexao.close()

In [ ]:
if not paciente:
        print("\n❌ Nome ou senha incorretos. Acesso negado.")
        input("\nPressione ENTER para voltar...")
        return

    p_id, p_nome, p_idade, p_queixa, p_dor, p_status = paciente


    while True:
        exibir_cabecalho(f"PAINEL DO PACIENTE - BEM-VINDO(A) {p_nome.upper()}")

        print(f"\n 📊 ESTATÍSTICAS ATUAIS:")
        print(f"    • Queixa/Sintoma : {p_queixa}")
        print(f"    • Nível de Dor   : {p_dor}/10")
        print(f"    • Status Triagem : {p_status}\n")

        print('-' * 65)
        print("  [ 1 ] Editar Estatísticas (Queixa e Nível de Dor)")
        print("  [ 2 ] Sair da conta\n")
        print('-' * 65)

        opcao = input("\nDigite o número da opção desejada: ").strip()

        if opcao == '1':
            exibir_cabecalho("EDITAR ESTATÍSTICAS DE SAÚDE")

            while True:
                nova_queixa = input("\nNova queixa/sintoma: ").strip()
                if nova_queixa:
                    break
                print("❌ Erro. A queixa não pode ficar em branco.")

            while True:
                try:
                    novo_niveldedor = int(input("Novo nível de dor (0 a 10): "))
                    if 0 <= novo_niveldedor <= 10:
                        break
                    print("❌ Erro. Digite um número entre 0 e 10.")
                except ValueError:
                    print("❌ Erro. Digite apenas números inteiros.")


            novo_status = 'Urgente' if novo_niveldedor >= 7 else 'Normal'


            cursor.execute("""
            UPDATE pacientes
            SET queixa = ?, niveldedor = ?, status = ?
            WHERE id = ?
            """, (nova_queixa, novo_niveldedor, novo_status, p_id))
            conexao.commit()


            p_queixa = nova_queixa
            p_dor = novo_niveldedor
            p_status = novo_status

            print("\n" + "=" * 65)
            print("Estatísticas atualizadas com sucesso no banco de dados!")
            print("=" * 65)
            input("\nPressione ENTER para continuar...")

        elif opcao == '2':
            print("\nSaindo da conta...")
            break
        else:
            input("\n❌ Opção inválida. Pressione ENTER para tentar novamente...")



while True:
    exibir_cabecalho("CENTRAL DE ATENDIMENTO - DR. COMMUNITY")

    print("\n  [ 1 ] Fazer um novo cadastro")
    print("  [ 2 ] Fazer login na conta")
    print("  [ 3 ] Ver cadastros realizados")
    print("  [ 4 ] Sair do programa\n")
    print('-' * 65)

    opcao = input("\nDigite o número da opção desejada: ").strip()

    match opcao:
        case '1':

            exibir_cabecalho("NOVO CADASTRO DE PACIENTE")

            while True:
                nome = input("\nQual é o seu nome completo? ").strip()
                if nome.replace(" ", "").isalpha():
                    break
                print("❌ Erro. Digite um nome válido, sem números ou caracteres especiais.")

            while True:
                senha = input("Crie uma senha de acesso: ").strip()
                if len(senha) >= 4:
                    break
                print("❌ Erro. A senha deve ter pelo menos 4 caracteres.")

            while True:
                try:
                    idade = int(input("Qual é a sua idade? "))
                    if idade >= 0:
                        break
                    print("❌ Erro. A idade deve ser maior ou igual a zero.")
                except ValueError:
                    print("❌ Erro. Digite a idade apenas em números.")

            while True:
                queixa = input("Qual é a sua queixa/sintoma? ").strip()
                if queixa:
                    break
                print("❌ Erro. A queixa não pode ficar em branco.")

            while True:
                try:
                    niveldedor = int(input("Qual é o seu nível de dor (0 a 10)? "))
                    if 0 <= niveldedor <= 10:
                        break
                    print("❌ Erro. O nível de dor deve ser um número entre 0 e 10.")
                except ValueError:
                    print("❌ Erro. Digite seu nível de dor apenas em números.")

            status = 'Urgente' if niveldedor >= 7 else 'Normal'

In [ ]:
class Atendimento:
    def cadastrar_paciente(self, nome, senha, idade, queixa, niveldedor, status):
        cursor.execute("""
        INSERT INTO pacientes (nome, senha, idade, queixa, niveldedor, status)
        VALUES (?, ?, ?, ?, ?, ?)
        """, (nome, senha, idade, queixa, niveldedor, status))
        conexao.commit()

    def listar_cadastros(self):
        cursor.execute("SELECT id, nome, idade, queixa, niveldedor, status FROM pacientes")
        return cursor.fetchall()


atendimento_service = Atendimento()


def limpar_tela():
    os.system('cls' if os.name == 'nt' else 'clear')


def exibir_cabecalho(titulo):
    limpar_tela()
    print('=' * 65)
    print(f"{titulo:^65}")
    print('=' * 65)


def realizar_login():
    exibir_cabecalho("LOGIN DO PACIENTE")

    nome = input("\nDigite seu nome registrado: ").strip()
    senha = input("Digite sua senha: ").strip()


    cursor.execute("""
    SELECT id, nome, idade, queixa, niveldedor, status
    FROM pacientes
    WHERE LOWER(nome) = LOWER(?) AND senha = ?
    """, (nome, senha))

    paciente = cursor.fetchone()


In [ ]:
import os
import sqlite3


conexao = sqlite3.connect('medconnect.db')
cursor = conexao.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS pacientes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL UNIQUE,
    senha TEXT NOT NULL,
    idade INTEGER NOT NULL,
    queixa TEXT NOT NULL,
    niveldedor INTEGER NOT NULL,
    status TEXT NOT NULL
)
""")
conexao.commit()


In [ ]:
import os
import sqlite3

conexao = sqlite3.connect('medconnect.db')
cursor = conexao.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS medicos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    crm TEXT NOT NULL UNIQUE,
    especialidade TEXT NOT NULL,
    senha TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS pacientes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    medico_id INTEGER NOT NULL,
    nome TEXT NOT NULL,
    idade INTEGER NOT NULL,
    queixa TEXT NOT NULL,
    niveldedor INTEGER NOT NULL,
    status TEXT NOT NULL,

    FOREIGN KEY (medico_id) REFERENCES medicos(id)
)
""")

conexao.commit()

class Atendimento:
    def cadastrar_medico(self, nome, crm, especialidade, senha):

        cursor.execute("""
        INSERT INTO medicos
        (nome, crm, especialidade, senha)
        VALUES (?, ?, ?, ?)
        """, (nome, crm, especialidade, senha))

        conexao.commit()

    def login_medico(self, crm, senha):

        cursor.execute("""
        SELECT id, nome, crm, especialidade
        FROM medicos
        WHERE crm = ? AND senha = ?
        """, (crm, senha))

        return cursor.fetchone()

    def cadastrar_paciente(
        self,
        medico_id,
        nome,
        idade,
        queixa,
        niveldedor,
        status
    ):

        cursor.execute("""
        INSERT INTO pacientes
        (medico_id, nome, idade, queixa, niveldedor, status)
        VALUES (?, ?, ?, ?, ?, ?)
        """, (
            medico_id,
            nome,
            idade,
            queixa,
            niveldedor,
            status
        ))

        conexao.commit()


    def listar_pacientes(self, medico_id):

        cursor.execute("""
        SELECT id, nome, idade, queixa, niveldedor, status
        FROM pacientes
        WHERE medico_id = ?
        ORDER BY niveldedor DESC
        """, (medico_id,))

        return cursor.fetchall()

    def buscar_paciente(self, paciente_id, medico_id):

        cursor.execute("""
        SELECT id, nome, idade, queixa, niveldedor, status
        FROM pacientes
        WHERE id = ? AND medico_id = ?
        """, (paciente_id, medico_id))

        return cursor.fetchone()

    def editar_paciente(
        self,
        paciente_id,
        medico_id,
        nome,
        idade,
        queixa,
        niveldedor,
        status
    ):

        cursor.execute("""
        UPDATE pacientes
        SET nome = ?,
            idade = ?,
            queixa = ?,
            niveldedor = ?,
            status = ?
        WHERE id = ? AND medico_id = ?
        """, (
            nome,
            idade,
            queixa,
            niveldedor,
            status,
            paciente_id,
            medico_id
        ))

        conexao.commit()

    def excluir_paciente(self, paciente_id, medico_id):

        cursor.execute("""
        DELETE FROM pacientes
        WHERE id = ? AND medico_id = ?
        """, (paciente_id, medico_id))

        conexao.commit()


atendimento_service = Atendimento()

def limpar_tela():

    os.system('cls' if os.name == 'nt' else 'clear')


def exibir_cabecalho(titulo):

    limpar_tela()

    print('='*80)
    print(f"{titulo:^70}")
    print('='*80)

def cadastrar_medico():

    exibir_cabecalho("CADASTRO DO MÉDICO")

    while True:

        nome = input(
            "\nDigite o nome completo do médico: "
        ).strip()

        if nome.replace(" ", "").isalpha():
            break

        print(
            "Erro. Digite um nome válido."
        )

    while True:

        crm = input(
            "Digite o CRM: "
        ).strip()

        if crm:
            break

        print(
            "Erro. O CRM não pode ficar vazio."
        )

    while True:

        especialidade = input(
            "Digite a especialidade: "
        ).strip()

        if especialidade:
            break

        print(
            "Erro. A especialidade não pode ficar vazia."
        )

    while True:

        senha = input(
            "Crie uma senha: "
        ).strip()

        if len(senha) >= 4:
            break

        print(
            "Erro. A senha deve ter pelo menos 4 caracteres."
        )

    try:

        atendimento_service.cadastrar_medico(
            nome,
            crm,
            especialidade,
            senha
        )


        print("MÉDICO CADASTRADO COM SUCESSO!")


        print(f"\nNome: {nome}")
        print(f"CRM: {crm}")
        print(f"Especialidade: {especialidade}")

    except sqlite3.IntegrityError:

        print(
            "\nErro: esse CRM já está cadastrado."
        )


    input(
        "\nDigite a tecla enter para voltar"
    )

def cadastrar_paciente(medico_id):

    exibir_cabecalho("CADASTRO DE NOVO PACIENTE")

    while True:

        nome = input(
            "\nNome completo do paciente: "
        ).strip()

        if nome.replace(" ", "").isalpha():
            break

        print(
            "Erro. Digite um nome válido."
        )

    while True:

        try:

            idade = int(
                input("Idade do paciente: ")
            )

            if idade >= 0:
                break

            print(
                "A idade não pode ser negativa."
            )

        except ValueError:

            print(
                "Digite a idade apenas em números."
            )

    while True:

        queixa = input(
            "Queixa/sintoma do paciente: "
        ).strip()

        if queixa:
            break

        print(
            "A queixa não pode ficar vazia."
        )

    while True:

        try:

            niveldedor = int(
                input("Nível de dor (0 a 10): ")
            )

            if 0 <= niveldedor <= 10:
                break

            print(
                "Digite um número entre 0 e 10."
            )

        except ValueError:

            print(
                "Digite apenas números."
            )

    if niveldedor >= 7:

        status = "Urgente"

    elif niveldedor >= 4:

        status = "Em observação"

    else:

        status = "Normal"

    atendimento_service.cadastrar_paciente(
        medico_id,
        nome,
        idade,
        queixa,
        niveldedor,
        status
    )

    print("PACIENTE CADASTRADO COM SUCESSO!")

    print(f"\nPaciente: {nome}")
    print(f"Idade: {idade}")
    print(f"Queixa: {queixa}")
    print(f"Nível de dor: {niveldedor}/10")
    print(f"Status: {status}")


    input("\nDigite a tecla enter para voltar")

def listar_pacientes(medico_id):

    exibir_cabecalho("MEUS PACIENTES")


    pacientes = atendimento_service.listar_pacientes(
        medico_id
    )


    if not pacientes:

        print(
            "\nVocê ainda não possui pacientes cadastrados."
        )

    else:

        print(
            f"\n{'ID':<5} | "
            f"{'NOME':<20} | "
            f"{'IDADE':<7} | "
            f"{'DOR':<5} | "
            f"{'STATUS':<15}"
        )



        for paciente in pacientes:

            p_id, p_nome, p_idade, p_queixa, p_dor, p_status = paciente

            print(
                f"{p_id:<5} | "
                f"{p_nome:<20} | "
                f"{p_idade:<7} | "
                f"{p_dor:<5} | "
                f"{p_status:<15}"
            )


    input(
        "\nDigite a tecla enter para voltar"
    )

def consultar_paciente(medico_id):

    exibir_cabecalho("CONSULTAR PACIENTE")


    try:

        paciente_id = int(
            input("\nDigite o ID do paciente: ")
        )

    except ValueError:

        print(
            "\nDigite um ID válido."
        )

        input(
            "\nPressione ENTER para voltar..."
        )

        return


    paciente = atendimento_service.buscar_paciente(
        paciente_id,
        medico_id
    )


    if not paciente:

        print(
            "\nPaciente não encontrado."
        )

    else:

        p_id, p_nome, p_idade, p_queixa, p_dor, p_status = paciente


        print("DADOS DO PACIENTE")
        print(f"\nID:              {p_id}")
        print(f"Nome:            {p_nome}")
        print(f"Idade:           {p_idade}")
        print(f"Queixa/Sintoma:  {p_queixa}")
        print(f"Nível de dor:    {p_dor}/10")
        print(f"Status:          {p_status}")


    input(
        "\nDigite a tecla enter para voltar"
    )

def painel_medico(medico):

    medico_id, medico_nome, medico_crm, especialidade = medico


    while True:

        exibir_cabecalho(
            f"PAINEL DO MÉDICO - DR(A). {medico_nome.upper()}"
        )


        print(f"\nMédico: {medico_nome}")
        print(f"CRM: {medico_crm}")
        print(f"Especialidade: {especialidade}")


        print("  [ 1 ] Cadastrar paciente")
        print("  [ 2 ] Ver meus pacientes")
        print("  [ 3 ] Consultar paciente")
        print("  [ 4 ] Sair da conta")




        opcao = input(
            "\nDigite o número da opção desejada: "
        ).strip()


        match opcao:

            case '1':

                cadastrar_paciente(medico_id)


            case '2':

                listar_pacientes(medico_id)


            case '3':

                consultar_paciente(medico_id)


            case '4':

                print(
                    "\nSaindo da conta..."
                )

                break


            case _:

                input(
                    "\nOpção inválida. "
                    "\nDigite a tecla enter para voltar"
                )

def login_medico():

    exibir_cabecalho("LOGIN DO MÉDICO")


    crm = input(
        "\nDigite seu CRM: "
    ).strip()


    senha = input(
        "Digite sua senha: "
    ).strip()


    medico = atendimento_service.login_medico(
        crm,
        senha
    )


    if not medico:

        print(
            "\nCRM ou senha incorretos."
        )

        input(
            "\nDigite a tecla enter para voltar"
        )

        return


    print(
        "\nLogin realizado com sucesso!"
    )


    input(
        "\nDigite a tecla enter para voltar"
    )


    painel_medico(medico)

while True:

    exibir_cabecalho(
        "---===DR.COMMUNITY===---"
    )


    print("\n  [ 1 ] Cadastrar como médico")
    print("  [ 2 ] Login do médico")
    print("  [ 3 ] Sair")


    opcao = input(
        "\nDigite o número da opção desejada: "
    ).strip()


    match opcao:

        case '1':

            cadastrar_medico()


        case '2':

            login_medico()


        case '3':

            print(
                "\nEncerrando o sistema... Até logo!"
            )

            break


        case _:

            input(
                "\nOpção inválida! "
                "\nDigite a tecla enter para voltar"
            )

conexao.close()

                       ---===DR.COMMUNITY===---                       

  [ 1 ] Cadastrar como médico
  [ 2 ] Login do médico
  [ 3 ] Sair
